# Wav2vec embeddings for ru-fr_interference
Output: `embeddings.npz` to download and feed it to local DVC pipeline for analysis

In [1]:
!pip -q install transformers torchaudio

In [2]:
import re
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, torch, torchaudio
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SPK_DIR = next(Path('/kaggle/input').rglob('FRcorp_textgrids_only'))
RUFR = next(Path('/kaggle/input').rglob('RUFRcorr.csv'))
print(SPK_DIR, '|', DEVICE)

/kaggle/input/datasets/alenamuravyeva/ru-fr-interference/ru-fr_interference/2/wav_et_textgrids/FRcorp_textgrids_only | cuda


In [3]:
targets = pd.read_csv(RUFR, sep='\t')
TARGETS = {row.Word: row.Word.lower().split() for _, row in targets.iterrows() if row.Word != 'distractor'}
print(f'{len(TARGETS)} target words')

12 target words


In [4]:
def read_csv(p):
    out = []
    for line in open(p, encoding='utf-8'):
        parts = line.strip().split(';')
        if len(parts) == 3 and parts[0].strip('"').strip():
            try: out.append((parts[0].strip('"').strip().lower(), float(parts[1]), float(parts[2])))
            except ValueError: pass
    return out

spk_tables = {}
for d in sorted(SPK_DIR.iterdir()):
    if not d.is_dir(): continue
    csvs = sorted([p for p in d.glob('*_words.csv') if re.search(r'FRcorp(\d+)', p.name)],
                  key=lambda p: int(re.search(r'FRcorp(\d+)', p.name).group(1)))
    table = []
    for ri, csvf in enumerate(csvs):
        wavf = csvf.with_name(csvf.name.replace('_words.csv', '.wav'))
        if wavf.exists():
            for w, t0, t1 in read_csv(csvf):
                table.append((w, t0, t1, wavf, ri))
    if table: spk_tables[d.name] = table
print(f'{len(spk_tables)} speakers')

19 speakers


In [5]:
matches = []
for spk, table in spk_tables.items():
    norm = [e[0] for e in table]
    for target, toks in TARGETS.items():
        L = len(toks)
        for i in range(len(table) - L + 1):
            if norm[i:i+L] == toks and len({table[i+k][4] for k in range(L)}) == 1:
                matches.append((spk, target, table[i][1], table[i+L-1][2], table[i][3]))
print(f'{len(matches)} matches')

1361 matches


In [6]:
fe = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')
model = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(DEVICE).eval()
SR, HOP = 16000, 0.020

by_wav = defaultdict(list)
for spk, w, t0, t1, wav in matches: by_wav[wav].append((spk, w, t0, t1))

embs, ws, sps = [], [], []
for wi, (wav_path, group) in enumerate(by_wav.items()):
    wav, sr = torchaudio.load(str(wav_path))
    if sr != SR: wav = torchaudio.functional.resample(wav, sr, SR)
    x = fe(wav.mean(0).numpy(), sampling_rate=SR, return_tensors='pt').input_values.to(DEVICE)
    with torch.no_grad():
        f = model(x).last_hidden_state[0].cpu().numpy().astype(np.float64)
    for spk, w, t0, t1 in group:
        f0, f1 = max(0, int(t0/HOP)), min(int(t1/HOP), f.shape[0])
        if f1 > f0:
            embs.append(f[f0:f1].mean(0)); ws.append(w); sps.append(spk)
    if (wi+1) % 50 == 0: print(f'{wi+1}/{len(by_wav)} wavs')

embs = np.stack(embs)
print(embs.shape)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

50/1361 wavs
100/1361 wavs
150/1361 wavs
200/1361 wavs
250/1361 wavs
300/1361 wavs
350/1361 wavs
400/1361 wavs
450/1361 wavs
500/1361 wavs
550/1361 wavs
600/1361 wavs
650/1361 wavs
700/1361 wavs
750/1361 wavs
800/1361 wavs
850/1361 wavs
900/1361 wavs
950/1361 wavs
1000/1361 wavs
1050/1361 wavs
1100/1361 wavs
1150/1361 wavs
1200/1361 wavs
1250/1361 wavs
1300/1361 wavs
1350/1361 wavs
(1361, 768)


In [7]:
np.savez('/kaggle/working/embeddings.npz', embs=embs, words=np.array(ws), speakers=np.array(sps))